In [1]:
1+1


2

In [2]:
import os

In [3]:
%pwd

'c:\\Users\\Priya\\Downloads\\Text-Summarizer\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Priya\\Downloads\\Text-Summarizer'

In [12]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [13]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [17]:
pip install transformers

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
Using cached huggingface_hub-1.7.1-py3-none-any.whl (616 kB)
Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl (3.7 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)

   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   

In [21]:
pip install datasets

  Using cached datasets-4.8.2-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.8.2-py3-none-any.whl (526 kB)
Using cached multiprocess-0.70.19-py313-none-any.whl (156 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl (27.5 MB)
Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl (31 kB)

  Attempting uninstall: pyarrow

    Found existing installation: pyarrow 19.0.0

   -------- ------------------------------- 1/5 [pyarrow]
   -------- ------------------------------- 1/5 [pyarrow]
    Uninstalling pyarrow-19.0.0:
   -------- ------------------------------- 1/5 [pyarrow]
      Successfully uninstalled pyarrow-19.0.0
   -------- --------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.7.0 requires pyarrow<23,>=4.0.0, but you have pyarrow 23.0.1 which is incompatible.


In [22]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

[2026-03-19 21:02:51,377: INFO: utils: NumExpr defaulting to 8 threads.]


In [23]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    
    def convert_examples_to_features(self,example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )
        
        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )
            
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [24]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-03-19 21:03:24,649: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-19 21:03:24,655: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-19 21:03:24,660: INFO: common: created directory at: artifacts]
[2026-03-19 21:03:24,664: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-19 21:03:25,778: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-19 21:03:25,847: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-19 21:03:26,174: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-03-19 21:03:26,177: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-03-19 21:03:26,211: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-03-19 21:03:26,551: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-03-19 21:03:26,833: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"]


Could not extract SentencePiece model from C:\Users\Priya\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail\snapshots\40d588fdab0cc077b80d950b300bf66ad3c75b92\spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.